ARC-AGI-2 notebook: **Bootstrap** (once) → **Config** (edit `PIPELINE` and knobs) → **Runner**.

- Set **`PIPELINE`** to one of: `validate_data`, `train`, `tune`, `submission`, `train_and_submit`, `tune_and_submit`. Other settings stay defined for every run so you can switch pipelines without missing variables.
- **`SUBMISSION_PIPELINE`**: submit `strategy` (e.g. `llm_tta_dfs` or `single`). **`LLM_*`** options feed `--llm-*` when strategy is `llm_tta_dfs`.
- **`DATA_ROOT`**: `None` uses framework default (Kaggle competition input or local `input/` layout). Set a string path to override.
- Submit stages log a **best-effort local eval score** when public eval JSON exists under `data_root`.


In [ ]:
# Bootstrap Cell: paths + framework + infra notebook API
#
# Must insert scripts/ before importing path_bootstrap (same resolution as
# path_bootstrap.resolve_notebook_scripts_path in the package).

import os
import sys

CONTEST = "arc_agi_2"
SCRIPTS_PACKAGE = "kaggle-ml-comp-scripts"

is_kaggle = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""
if is_kaggle:
    scripts_candidates = [
        f"/kaggle/input/datasets/mcusac/{SCRIPTS_PACKAGE}/scripts",
        f"/kaggle/input/{SCRIPTS_PACKAGE}/scripts",
    ]
else:
    scripts_candidates = [
        f"../input/{SCRIPTS_PACKAGE}/scripts",
        f"../{SCRIPTS_PACKAGE}/scripts",
    ]

scripts_path = next((p for p in scripts_candidates if os.path.isdir(p)), None)
if scripts_path is None:
    raise FileNotFoundError(
        "Could not locate kaggle-ml-comp-scripts/scripts. Checked:\n"
        + "\n".join(f" - {p}" for p in scripts_candidates)
    )

if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from path_bootstrap import bootstrap_notebook_environment

scripts_path = bootstrap_notebook_environment(contest=CONTEST, scripts_package=SCRIPTS_PACKAGE)

# Side effect: registers notebook_commands + contest handlers for this process (notebooks do not run run.py).
import layers.layer_1_competition.level_1_impl.level_arc_agi_2.registration  # noqa: F401

from layers.layer_1_competition.level_0_infra.level_2.notebook import (
    get_notebook_commands_module,
    list_contests_with_notebook_commands,
    run_cli_streaming,
)

available_contests = list_contests_with_notebook_commands()

try:
    cmds = get_notebook_commands_module(CONTEST)
except ValueError as exc:
    raise ValueError(
        f"Contest '{CONTEST}' is not registered for notebook commands. "
        f"Available contests: {available_contests}. "
        "Import the contest registration module before get_notebook_commands_module, "
        "or change CONTEST to an available key."
    ) from exc

print("✅ Setup complete. Run Config Cell next.")
print(f"   Contest: {CONTEST}")
print(f"   Scripts path: {scripts_path}")
print(f"   Contests with notebook_commands registered: {available_contests}")


In [ ]:
# Config Cell — edit this block, then run Runner
import math
import os

PIPELINE = "submission"
PIPELINE_CHOICES = (
    "validate_data",
    "train",
    "tune",
    "submission",
    "train_and_submit",
    "tune_and_submit",
)

DATA_ROOT = None
SMOKE_MAX_TARGETS = 3
VALIDATE_LOG_FILE = None

TRAIN_PIPELINE = "end_to_end"
TRAIN_MODELS = ["baseline_approx"]

TUNE_MODEL = "baseline_approx"
TUNE_SEARCH_TYPE = "quick"

SUBMISSION_PIPELINE = "llm_tta_dfs"
SUBMISSION_MODELS = ["baseline_approx"]
SUBMISSION_ENSEMBLE_WEIGHTS = None
USE_VALIDATION_FOR_STACKING = True

LLM_EXECUTION_MODE = "lm_backend"
LLM_NUM_AUGMENTATIONS = 8
LLM_BEAM_WIDTH = 12
LLM_MAX_CANDIDATES = 6
LLM_MAX_NEG_LOG_SCORE = -math.log(0.2)
LLM_SEED = 0

if PIPELINE not in PIPELINE_CHOICES:
    raise ValueError(f"PIPELINE must be one of {PIPELINE_CHOICES}, got {PIPELINE!r}")

_is_kaggle = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""
if _is_kaggle:
    _llm_dir_candidates = [
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    ]
else:
    _env_model = (os.environ.get("ARC_LLM_MODEL_PATH") or "").strip()
    _llm_dir_candidates = [
        _env_model,
        "../input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    ]
LLM_MODEL_PATH = next((p for p in _llm_dir_candidates if p and os.path.isdir(p)), None)
_env_lora = (os.environ.get("ARC_LLM_LORA_PATH") or "").strip()
LLM_LORA_PATH = _env_lora if _env_lora and os.path.isdir(_env_lora) else None

LLM_SUBMIT_KWARGS = dict(
    llm_execution_mode=LLM_EXECUTION_MODE,
    llm_num_augmentations=LLM_NUM_AUGMENTATIONS,
    llm_beam_width=LLM_BEAM_WIDTH,
    llm_max_candidates=LLM_MAX_CANDIDATES,
    llm_max_neg_log_score=LLM_MAX_NEG_LOG_SCORE,
    llm_seed=LLM_SEED,
    llm_model_path=LLM_MODEL_PATH,
    llm_lora_path=LLM_LORA_PATH,
)

_cfg_lines = [
    f"PIPELINE: {PIPELINE}",
    f"DATA_ROOT: {DATA_ROOT!r}",
    f"SMOKE_MAX_TARGETS: {SMOKE_MAX_TARGETS}",
]
if PIPELINE == "validate_data":
    _cfg_lines.append(f"VALIDATE_LOG_FILE: {VALIDATE_LOG_FILE}")
if PIPELINE in ("train", "train_and_submit"):
    _cfg_lines += [f"TRAIN_PIPELINE: {TRAIN_PIPELINE}", f"TRAIN_MODELS: {TRAIN_MODELS}"]
if PIPELINE in ("tune", "tune_and_submit"):
    _cfg_lines += [f"TUNE_MODEL: {TUNE_MODEL}", f"TUNE_SEARCH_TYPE: {TUNE_SEARCH_TYPE}"]
if PIPELINE in ("submission", "train_and_submit", "tune_and_submit"):
    _cfg_lines += [
        f"SUBMISSION_PIPELINE: {SUBMISSION_PIPELINE}",
        f"SUBMISSION_MODELS: {SUBMISSION_MODELS}",
        f"LLM_MODEL_PATH: {LLM_MODEL_PATH!r}",
        f"LLM_LORA_PATH: {LLM_LORA_PATH!r}",
    ]
    if SUBMISSION_PIPELINE == "llm_tta_dfs" and LLM_EXECUTION_MODE == "lm_backend" and not LLM_MODEL_PATH:
        _cfg_lines.append(
            "⚠️ lm_backend needs a model dir. Set ARC_LLM_MODEL_PATH or add the Qwen bundle under input/."
        )
print("Config ready\n   " + "\n   ".join(_cfg_lines))

In [ ]:
if PIPELINE == "validate_data":
    cmd = cmds.build_validate_data_command(
        data_root=DATA_ROOT,
        max_targets=SMOKE_MAX_TARGETS,
        log_file=VALIDATE_LOG_FILE,
    )
    description = f"{CONTEST} validate_data (max_targets={SMOKE_MAX_TARGETS})"

elif PIPELINE == "train":
    cmd = cmds.build_train_command(
        data_root=DATA_ROOT,
        train_mode=TRAIN_PIPELINE,
        models=TRAIN_MODELS,
    )
    description = f"Training {TRAIN_PIPELINE} models: {TRAIN_MODELS}"

elif PIPELINE == "tune":
    cmd = cmds.build_tune_command(
        data_root=DATA_ROOT,
        model=TUNE_MODEL,
        search_type=TUNE_SEARCH_TYPE,
        max_targets=SMOKE_MAX_TARGETS,
    )
    description = f"Tuning {TUNE_MODEL} with {TUNE_SEARCH_TYPE} search"

elif PIPELINE in ("submission", "train_and_submit", "tune_and_submit"):
    submit_common = dict(
        data_root=DATA_ROOT,
        strategy=SUBMISSION_PIPELINE,
        max_targets=SMOKE_MAX_TARGETS,
        ensemble_weights=SUBMISSION_ENSEMBLE_WEIGHTS,
        use_validation_for_stacking=USE_VALIDATION_FOR_STACKING,
        **LLM_SUBMIT_KWARGS,
    )
    if PIPELINE == "submission":
        cmd = cmds.build_submit_command(models=SUBMISSION_MODELS, **submit_common)
        description = f"Submission {SUBMISSION_PIPELINE} models={SUBMISSION_MODELS}"
    elif PIPELINE == "train_and_submit":
        cmd = cmds.build_train_and_submit_command(
            train_mode=TRAIN_PIPELINE,
            models=TRAIN_MODELS,
            **submit_common,
        )
        description = f"train_and_submit train={TRAIN_PIPELINE} submit={SUBMISSION_PIPELINE}"
    else:
        cmd = cmds.build_tune_and_submit_command(
            model=TUNE_MODEL,
            search_type=TUNE_SEARCH_TYPE,
            models=SUBMISSION_MODELS,
            **submit_common,
        )
        description = f"tune_and_submit tune={TUNE_MODEL} submit={SUBMISSION_PIPELINE}"

else:
    raise ValueError(f"Unknown PIPELINE: {PIPELINE}")

returncode, stdout_lines = run_cli_streaming(cmd, description=description, keep_last_n=200)

if returncode != 0:
    raise RuntimeError(f"{PIPELINE} pipeline failed. See output above.\n" + "\n".join(stdout_lines[-50:]))
